---

# <span style="color:#1F4E79;">25340 Digital Ocean</span>
### <span style="color:#2E86AB;">Sea Surface Temperature Anomalies and Atmospheric Drivers of the North Atlantic Marine Heatwaves</span>

---

In [1]:
# Import necessary libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import dates as mdates
import copernicusmarine
import warnings
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output
from datetime import datetime
import calendar
from pathlib import Path
import os
from getpass import getpass
import cdsapi
import xarray as xr

c:\Users\natal\repos\OceanCourses\.ocean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


login: nslimak

password: SzybkieRybki123!

### Sea Surface Temperature (SST)

In [4]:
# Common request settings (smaller area = faster download)
common_request = {
    "dataset_id": "C3S-GLO-SST-L4-REP-OBS-SST",
    "dataset_version": "202506",
    "variables": ["analysed_sst"],
    "minimum_longitude": -80.0,
    "maximum_longitude": 0.0,
    "minimum_latitude": 20.0,
    "maximum_latitude": 65.0,
    "output_directory": "Data",
    "overwrite": True,
}

# Two target periods
periods = [
    {
        "name": "mhw_2016",
        "start": "2015-12-01T00:00:00",
        "end": "2016-04-30T23:59:59",
        "output": "atl_sst_mhw_2016.nc",
    },
    {
        "name": "mhw_2023",
        "start": "2023-05-01T00:00:00",
        "end": "2023-09-30T23:59:59",
        "output": "atl_sst_mhw_2023.nc",
    },
]

for p in periods:
    print(f"Downloading {p['name']}...")
    try:
        copernicusmarine.subset(
            **common_request,
            start_datetime=p["start"],
            end_datetime=p["end"],
            output_filename=p["output"],
        )
        print(f"Done: {p['output']}")
    except Exception as e:
        print(f"Skipped {p['name']}: {e}")

print("Finished all period requests.")

INFO - 2026-03-27T20:42:15Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:Copernicus Marine password:

INFO - 2026-03-27T20:42:31Z - Selected dataset version: "202506"
INFO - 2026-03-27T20:42:31Z - Selected dataset part: "default"
100%|██████████| 341/341 [05:31<00:00,  1.03it/s]
INFO - 2026-03-27T20:48:12Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Done: atl_sst_mhw_2016.nc
Copernicus Marine username:Copernicus Marine password:

INFO - 2026-03-27T20:48:20Z - Selected dataset version: "202506"
INFO - 2026-03-27T20:48:20Z - Selected dataset part: "default"
100%|██████████| 341/341 [05:11<00:00,  1.10it/s]

Done: atl_sst_mhw_2023.nc
Finished all period requests.


In [1]:
# ERA5 derived daily statistics (6-hour frequency) via CDS API

from datetime import datetime
import calendar
from pathlib import Path
import os
from getpass import getpass
import cdsapi
import xarray as xr

# Area of interest: North, West, South, East
area = [-80, 0, 20, 65]

# Exactly two outputs
periods = [
    {"start": "2015-12-01", "end": "2016-04-30", "output": "era5_tcc_mhw_2016.nc"},
    {"start": "2023-05-01", "end": "2023-09-30", "output": "era5_tcc_mhw_2023.nc"},
]

# CDS key can be overridden by .cdsapirc or CDSAPI_KEY env var
HARDCODED_CDS_KEY = "c3e3c608-3dad-41cd-bc47-98685785ad17"


def iter_months(start_date: datetime, end_date: datetime):
    current = datetime(start_date.year, start_date.month, 1)
    while current <= end_date:
        yield current
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)


def ensure_cds_credentials():
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return
    if os.getenv("CDSAPI_KEY") or HARDCODED_CDS_KEY:
        return

    print("CDS credentials not found.")
    print("Paste your CDS API key from your CDS profile page.")
    print("Format: <UID>:<API_KEY>")
    key_value = getpass("CDS key (input hidden): ").strip()
    if not key_value or ":" not in key_value:
        raise RuntimeError("Invalid CDS key format. Expected <UID>:<API_KEY>.")

    cds_rc.write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {key_value}\n",
        encoding="utf-8",
    )
    print(f"Created credentials file at: {cds_rc}")


def build_cds_client():
    ensure_cds_credentials()
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return cdsapi.Client()

    cds_url = os.getenv("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")
    cds_key = os.getenv("CDSAPI_KEY", HARDCODED_CDS_KEY)
    if cds_key:
        return cdsapi.Client(url=cds_url, key=cds_key)

    raise RuntimeError("CDS credentials are not available.")


client = build_cds_client()
out_dir = Path("Data")
tmp_dir = out_dir
out_dir.mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(parents=True, exist_ok=True)

created_files = []

for p in periods:
    start_dt = datetime.fromisoformat(p["start"])
    end_dt = datetime.fromisoformat(p["end"])
    prefix = Path(p["output"]).stem

    monthly_files = []
    for month_dt in iter_months(start_dt, end_dt):
        year = f"{month_dt.year:04d}"
        month = f"{month_dt.month:02d}"
        ndays = calendar.monthrange(month_dt.year, month_dt.month)[1]

        first_day = start_dt.day if (month_dt.year == start_dt.year and month_dt.month == start_dt.month) else 1
        last_day = end_dt.day if (month_dt.year == end_dt.year and month_dt.month == end_dt.month) else ndays

        days = [f"{d:02d}" for d in range(first_day, last_day + 1)]
        month_file = tmp_dir / f"{prefix}_{year}_{month}.nc"
        print(f"Downloading {month_file.name} ...")

        client.retrieve(
            "derived-era5-single-levels-daily-statistics",
            {
                "product_type": "reanalysis",
                "variable": "total_cloud_cover",
                "year": year,
                "month": month,
                "day": days,
                "daily_statistic": "daily_mean",
                "time_zone": "utc+00:00",
                "frequency": "6_hourly",
                "area": area,
                "format": "netcdf",
            },
            str(month_file),
        )
        monthly_files.append(month_file)

    datasets = [xr.open_dataset(fp) for fp in monthly_files]
    try:
        combined = xr.concat(datasets, dim="time").sortby("time")
        target_file = out_dir / p["output"]
        combined.to_netcdf(target_file)
        created_files.append(target_file.name)
        print(f"Created: {target_file.name}")
        combined.close()
    finally:
        for ds in datasets:
            ds.close()
        for fp in monthly_files:
            if fp.exists():
                fp.unlink()

print("Done. Output files:")
for f in created_files:
    print(f" - {f}")

2026-04-04 17:01:47,602 INFO Request ID is 8a5bcade-d80b-4f9f-b8fe-fa2b14bd7358
2026-04-04 17:01:47,694 INFO status has been updated to accepted
2026-04-04 17:02:01,344 INFO status has been updated to successful


2026-04-04 17:02:06,112 INFO Request ID is 765a0f15-96d9-4542-911c-1ca5578d54a5
2026-04-04 17:02:06,203 INFO status has been updated to accepted
2026-04-04 17:02:27,520 INFO status has been updated to successful


2026-04-04 17:02:33,740 INFO Request ID is edb34011-cdee-4e39-9396-47ab5d51fb89
2026-04-04 17:02:33,822 INFO status has been updated to accepted
2026-04-04 17:02:47,454 INFO status has been updated to running
2026-04-04 17:02:55,155 INFO status has been updated to successful


2026-04-04 17:02:58,766 INFO Request ID is 9593f359-55a5-4b22-82ee-efbc61a7cfad
2026-04-04 17:02:58,846 INFO status has been updated to accepted
2026-04-04 17:03:12,444 INFO status has been updated to running
2026-04-04 17:03:20,217 INFO status has been updated to successful


2026-04-04 17:03:23,849 INFO Request ID is 5de1b74e-c3d2-4784-9223-95347bdf9c44
2026-04-04 17:03:23,936 INFO status has been updated to accepted
2026-04-04 17:04:13,856 INFO status has been updated to successful
C:\Users\natal\AppData\Local\Temp\ipykernel_6212\3295480177.py:116: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  combined = xr.concat(datasets, dim="time").sortby("time")


...

2026-04-04 17:05:07,659 INFO Request ID is 8f23fb7e-515c-409f-ac5f-3b953538bcf5
2026-04-04 17:05:07,730 INFO status has been updated to accepted
2026-04-04 17:05:29,360 INFO status has been updated to running
2026-04-04 17:05:58,005 INFO status has been updated to successful


2026-04-04 17:06:12,040 INFO Request ID is 23813854-78d9-4a67-881a-f2144e43e3ee
2026-04-04 17:06:12,127 INFO status has been updated to accepted
2026-04-04 17:06:25,722 INFO status has been updated to running
2026-04-04 17:06:44,894 INFO status has been updated to successful


2026-04-04 17:06:53,964 INFO Request ID is 565f195e-2dad-4693-bf3c-1d33408e6c87
2026-04-04 17:06:54,042 INFO status has been updated to accepted
2026-04-04 17:07:15,384 INFO status has been updated to running
2026-04-04 17:07:44,022 INFO status has been updated to successful


2026-04-04 17:07:48,625 INFO Request ID is 468668d8-9349-43df-94b5-598e2c0a0193
2026-04-04 17:07:48,714 INFO status has been updated to accepted
2026-04-04 17:08:02,412 INFO status has been updated to running
2026-04-04 17:08:21,558 INFO status has been updated to successful
C:\Users\natal\AppData\Local\Temp\ipykernel_6212\3295480177.py:116: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  combined = xr.concat(datasets, dim="time").sortby("time")


Created: era5_tcc_mhw_2023.nc
Done. Output files:
 - era5_tcc_mhw_2016.nc
 - era5_tcc_mhw_2023.nc


In [3]:
# ERA5 derived daily statistics (6-hour frequency) via CDS API

from datetime import datetime
import calendar
from pathlib import Path
import os
from getpass import getpass
import cdsapi
import xarray as xr

# Area of interest: North, West, South, East
area = [-80, 0, 20, 65]

# Exactly two outputs
periods = [
    {"start": "2015-12-01", "end": "2016-04-30", "output": "era5_u-wind_mhw_2016.nc"},
    {"start": "2023-05-01", "end": "2023-09-30", "output": "era5_u-wind_mhw_2023.nc"},
]

# CDS key can be overridden by .cdsapirc or CDSAPI_KEY env var
HARDCODED_CDS_KEY = "c3e3c608-3dad-41cd-bc47-98685785ad17"


def iter_months(start_date: datetime, end_date: datetime):
    current = datetime(start_date.year, start_date.month, 1)
    while current <= end_date:
        yield current
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)


def ensure_cds_credentials():
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return
    if os.getenv("CDSAPI_KEY") or HARDCODED_CDS_KEY:
        return

    print("CDS credentials not found.")
    print("Paste your CDS API key from your CDS profile page.")
    print("Format: <UID>:<API_KEY>")
    key_value = getpass("CDS key (input hidden): ").strip()
    if not key_value or ":" not in key_value:
        raise RuntimeError("Invalid CDS key format. Expected <UID>:<API_KEY>.")

    cds_rc.write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {key_value}\n",
        encoding="utf-8",
    )
    print(f"Created credentials file at: {cds_rc}")


def build_cds_client():
    ensure_cds_credentials()
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return cdsapi.Client()

    cds_url = os.getenv("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")
    cds_key = os.getenv("CDSAPI_KEY", HARDCODED_CDS_KEY)
    if cds_key:
        return cdsapi.Client(url=cds_url, key=cds_key)

    raise RuntimeError("CDS credentials are not available.")


client = build_cds_client()
out_dir = Path("Data")
tmp_dir = out_dir
out_dir.mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(parents=True, exist_ok=True)

created_files = []

for p in periods:
    start_dt = datetime.fromisoformat(p["start"])
    end_dt = datetime.fromisoformat(p["end"])
    prefix = Path(p["output"]).stem

    monthly_files = []
    for month_dt in iter_months(start_dt, end_dt):
        year = f"{month_dt.year:04d}"
        month = f"{month_dt.month:02d}"
        ndays = calendar.monthrange(month_dt.year, month_dt.month)[1]

        first_day = start_dt.day if (month_dt.year == start_dt.year and month_dt.month == start_dt.month) else 1
        last_day = end_dt.day if (month_dt.year == end_dt.year and month_dt.month == end_dt.month) else ndays

        days = [f"{d:02d}" for d in range(first_day, last_day + 1)]
        month_file = tmp_dir / f"{prefix}_{year}_{month}.nc"
        print(f"Downloading {month_file.name} ...")

        client.retrieve(
            "derived-era5-single-levels-daily-statistics",
            {
                "product_type": "reanalysis",
                "variable": "100m_u_component_of_wind",
                "year": year,
                "month": month,
                "day": days,
                "daily_statistic": "daily_mean",
                "time_zone": "utc+00:00",
                "frequency": "6_hourly",
                "area": area,
                "format": "netcdf",
            },
            str(month_file),
        )
        monthly_files.append(month_file)

    datasets = [xr.open_dataset(fp) for fp in monthly_files]
    try:
        combined = xr.concat(datasets, dim="time").sortby("time")
        target_file = out_dir / p["output"]
        combined.to_netcdf(target_file)
        created_files.append(target_file.name)
        print(f"Created: {target_file.name}")
        combined.close()
    finally:
        for ds in datasets:
            ds.close()
        for fp in monthly_files:
            if fp.exists():
                fp.unlink()

print("Done. Output files:")
for f in created_files:
    print(f" - {f}")

2026-04-04 17:14:54,564 INFO Request ID is 78b926c4-0b74-423e-87c9-876aa31e5eb4
2026-04-04 17:14:54,656 INFO status has been updated to accepted
2026-04-04 17:15:03,125 INFO status has been updated to running
2026-04-04 17:15:27,427 INFO status has been updated to successful


2026-04-04 17:15:33,779 INFO Request ID is 728f1c92-0c43-430b-a840-9c44af9ce47b
2026-04-04 17:15:33,888 INFO status has been updated to accepted
2026-04-04 17:15:55,137 INFO status has been updated to running
2026-04-04 17:16:06,604 INFO status has been updated to successful


2026-04-04 17:16:10,731 INFO Request ID is 70eefc17-dc29-41b1-a00b-3f2b3a9d6c47
2026-04-04 17:16:10,801 INFO status has been updated to accepted
2026-04-04 17:16:19,280 INFO status has been updated to running
2026-04-04 17:16:43,573 INFO status has been updated to successful


2026-04-04 17:16:47,550 INFO Request ID is 957a3c5b-9842-4018-9db6-57e6cbe72ccd
2026-04-04 17:16:48,127 INFO status has been updated to accepted
2026-04-04 17:17:01,701 INFO status has been updated to running
2026-04-04 17:17:20,895 INFO status has been updated to successful


2026-04-04 17:17:26,790 INFO Request ID is 29b65e41-150a-4961-81fa-9b825e00f776
2026-04-04 17:17:26,876 INFO status has been updated to accepted
2026-04-04 17:17:48,167 INFO status has been updated to running
2026-04-04 17:18:16,812 INFO status has been updated to successful
C:\Users\natal\AppData\Local\Temp\ipykernel_6212\1154665836.py:116: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  combined = xr.concat(datasets, dim="time").sortby("time")


...

2026-04-04 17:19:01,842 INFO Request ID is 5ce87b7c-4972-4289-8e91-b77499996f62
2026-04-04 17:19:01,919 INFO status has been updated to accepted
2026-04-04 17:19:15,536 INFO status has been updated to running
2026-04-04 17:19:51,885 INFO status has been updated to successful


2026-04-04 17:19:56,577 INFO Request ID is 31e27f97-b9c0-4083-b436-50b6a44f4815
2026-04-04 17:19:56,679 INFO status has been updated to accepted
2026-04-04 17:20:05,161 INFO status has been updated to running
2026-04-04 17:20:46,094 INFO status has been updated to successful


2026-04-04 17:20:53,392 INFO Request ID is 8c5eeadd-5937-4f2c-b732-fc2679ef656b
2026-04-04 17:20:53,465 INFO status has been updated to accepted
2026-04-04 17:21:14,766 INFO status has been updated to running
2026-04-04 17:21:43,452 INFO status has been updated to successful


2026-04-04 17:21:50,750 INFO Request ID is c6c9af3e-2714-4c17-812c-3dd1c0d5efc1
2026-04-04 17:21:50,851 INFO status has been updated to accepted
2026-04-04 17:21:59,350 INFO status has been updated to running
2026-04-04 17:22:23,674 INFO status has been updated to successful
C:\Users\natal\AppData\Local\Temp\ipykernel_6212\1154665836.py:116: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  combined = xr.concat(datasets, dim="time").sortby("time")


Created: era5_u-wind_mhw_2023.nc
Done. Output files:
 - era5_u-wind_mhw_2016.nc
 - era5_u-wind_mhw_2023.nc


In [4]:
# ERA5 derived daily statistics (6-hour frequency) via CDS API

from datetime import datetime
import calendar
from pathlib import Path
import os
from getpass import getpass
import cdsapi
import xarray as xr

# Area of interest: North, West, South, East
area = [-80, 0, 20, 65]

# Exactly two outputs
periods = [
    {"start": "2015-12-01", "end": "2016-04-30", "output": "era5_v-wind_mhw_2016.nc"},
    {"start": "2023-05-01", "end": "2023-09-30", "output": "era5_v-wind_mhw_2023.nc"},
]

# CDS key can be overridden by .cdsapirc or CDSAPI_KEY env var
HARDCODED_CDS_KEY = "c3e3c608-3dad-41cd-bc47-98685785ad17"


def iter_months(start_date: datetime, end_date: datetime):
    current = datetime(start_date.year, start_date.month, 1)
    while current <= end_date:
        yield current
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)


def ensure_cds_credentials():
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return
    if os.getenv("CDSAPI_KEY") or HARDCODED_CDS_KEY:
        return

    print("CDS credentials not found.")
    print("Paste your CDS API key from your CDS profile page.")
    print("Format: <UID>:<API_KEY>")
    key_value = getpass("CDS key (input hidden): ").strip()
    if not key_value or ":" not in key_value:
        raise RuntimeError("Invalid CDS key format. Expected <UID>:<API_KEY>.")

    cds_rc.write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {key_value}\n",
        encoding="utf-8",
    )
    print(f"Created credentials file at: {cds_rc}")


def build_cds_client():
    ensure_cds_credentials()
    cds_rc = Path.home() / ".cdsapirc"
    if cds_rc.exists():
        return cdsapi.Client()

    cds_url = os.getenv("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")
    cds_key = os.getenv("CDSAPI_KEY", HARDCODED_CDS_KEY)
    if cds_key:
        return cdsapi.Client(url=cds_url, key=cds_key)

    raise RuntimeError("CDS credentials are not available.")


client = build_cds_client()
out_dir = Path("Data")
tmp_dir = out_dir
out_dir.mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(parents=True, exist_ok=True)

created_files = []

for p in periods:
    start_dt = datetime.fromisoformat(p["start"])
    end_dt = datetime.fromisoformat(p["end"])
    prefix = Path(p["output"]).stem

    monthly_files = []
    for month_dt in iter_months(start_dt, end_dt):
        year = f"{month_dt.year:04d}"
        month = f"{month_dt.month:02d}"
        ndays = calendar.monthrange(month_dt.year, month_dt.month)[1]

        first_day = start_dt.day if (month_dt.year == start_dt.year and month_dt.month == start_dt.month) else 1
        last_day = end_dt.day if (month_dt.year == end_dt.year and month_dt.month == end_dt.month) else ndays

        days = [f"{d:02d}" for d in range(first_day, last_day + 1)]
        month_file = tmp_dir / f"{prefix}_{year}_{month}.nc"
        print(f"Downloading {month_file.name} ...")

        client.retrieve(
            "derived-era5-single-levels-daily-statistics",
            {
                "product_type": "reanalysis",
                "variable": "100m_v_component_of_wind",
                "year": year,
                "month": month,
                "day": days,
                "daily_statistic": "daily_mean",
                "time_zone": "utc+00:00",
                "frequency": "6_hourly",
                "area": area,
                "format": "netcdf",
            },
            str(month_file),
        )
        monthly_files.append(month_file)

    datasets = [xr.open_dataset(fp) for fp in monthly_files]
    try:
        combined = xr.concat(datasets, dim="time").sortby("time")
        target_file = out_dir / p["output"]
        combined.to_netcdf(target_file)
        created_files.append(target_file.name)
        print(f"Created: {target_file.name}")
        combined.close()
    finally:
        for ds in datasets:
            ds.close()
        for fp in monthly_files:
            if fp.exists():
                fp.unlink()

print("Done. Output files:")
for f in created_files:
    print(f" - {f}")

2026-04-04 17:33:06,247 INFO Request ID is b7f71e12-0762-4738-8682-d98dd715fa40
2026-04-04 17:33:06,340 INFO status has been updated to accepted
2026-04-04 17:33:19,993 INFO status has been updated to running
2026-04-04 17:33:39,192 INFO status has been updated to successful


2026-04-04 17:33:43,667 INFO Request ID is 3a3878ae-cd44-43c9-9a33-f95685f838cd
2026-04-04 17:33:43,743 INFO status has been updated to accepted
2026-04-04 17:34:05,050 INFO status has been updated to running
2026-04-04 17:34:16,547 INFO status has been updated to successful


2026-04-04 17:34:23,268 INFO Request ID is 250b71a9-dbdc-439e-85b3-108f55fe53e4
2026-04-04 17:34:23,362 INFO status has been updated to accepted
2026-04-04 17:34:36,985 INFO status has been updated to running
2026-04-04 17:34:56,149 INFO status has been updated to successful


2026-04-04 17:35:00,755 INFO Request ID is 7dde63e5-5b7e-4f91-a1e8-b1c3b1243664
2026-04-04 17:35:00,836 INFO status has been updated to accepted
2026-04-04 17:35:14,565 INFO status has been updated to running
2026-04-04 17:35:33,741 INFO status has been updated to successful


2026-04-04 17:35:38,336 INFO Request ID is 01d006d5-adde-4a87-b2f2-a0578c58ac53
2026-04-04 17:35:38,420 INFO status has been updated to accepted
2026-04-04 17:35:52,084 INFO status has been updated to running
2026-04-04 17:36:11,225 INFO status has been updated to successful
C:\Users\natal\AppData\Local\Temp\ipykernel_6212\1043235693.py:116: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  combined = xr.concat(datasets, dim="time").sortby("time")


...

2026-04-04 17:37:22,529 INFO Request ID is ac21422f-55e6-4df8-9ccc-d68cdf40a5ae
2026-04-04 17:37:22,623 INFO status has been updated to accepted
2026-04-04 17:37:36,322 INFO status has been updated to running
2026-04-04 17:37:55,523 INFO status has been updated to successful


2026-04-04 17:38:01,849 INFO Request ID is ba711bd3-6793-42e0-9914-0611a9695b62
2026-04-04 17:38:01,929 INFO status has been updated to accepted
2026-04-04 17:38:23,230 INFO status has been updated to running
2026-04-04 17:38:34,721 INFO status has been updated to successful


2026-04-04 17:38:38,445 INFO Request ID is eef2f7fe-766e-4755-8876-4521078ae70e
2026-04-04 17:38:38,538 INFO status has been updated to accepted
2026-04-04 17:38:47,062 INFO status has been updated to running
2026-04-04 17:39:11,393 INFO status has been updated to successful


2026-04-04 17:39:16,185 INFO Request ID is 2ababe74-ff5f-4327-ac08-189cedf8102b
2026-04-04 17:39:16,558 INFO status has been updated to accepted
2026-04-04 17:39:37,852 INFO status has been updated to running
2026-04-04 17:39:49,338 INFO status has been updated to successful
C:\Users\natal\AppData\Local\Temp\ipykernel_6212\1043235693.py:116: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  combined = xr.concat(datasets, dim="time").sortby("time")


Created: era5_v-wind_mhw_2023.nc
Done. Output files:
 - era5_v-wind_mhw_2016.nc
 - era5_v-wind_mhw_2023.nc
